In [ ]:
-- LC-1757
select product_id from products where low_fats = "Y" and recyclable = "Y";

In [ ]:
-- LC 584
SELECT name
FROM Customer
WHERE referee_id != 2 OR referee_id is null;

In [ ]:
-- LC 595
SELECT name,population,area
FROM World
where area >=3000000 or population >= 25000000;

In [ ]:
-- lc 1148
SELECT distinct author_id as 'id'
FROM Views
WHERE
author_id = viewer_id
order by id asc;

In [ ]:
--  lc 1683
select tweet_id
from Tweets
where char_length(content) > 15;

In [ ]:
--  lc 1378
select EmployeeUNI.unique_id,Employees.name
from Employees
left join EmployeeUNI
on Employees.id = EmployeeUNI.id;

In [ ]:
select Product.product_name, Sales.year,Sales.price
from Sales
left join Product
on Sales.product_id = Product.product_id;

In [ ]:
-- LC 1581 IMPORTANT
select v.customer_id,count(*) as count_no_trans
from Visits v
left join Transactions t
on v.visit_id = t.visit_id
where transaction_id is null
group by customer_id

In [ ]:
-- ḶC 197 IMPO
select w1.id
from weather w1
inner join weather w2
on w1.recordDate = date_add(w2.recordDate, interval 1 day)
where w1.temperature > w2.temperature

In [ ]:
-- lc 1661 imp
select a1.machine_id,round(avg(a2.timestamp-a1.timestamp),3) as processing_time
from Activity a1
join Activity a2
on a1.process_id = a2.process_id
and a1.machine_id = a2.machine_id
and a1.activity_type = 'start'
and a2.activity_type = 'end'
group by a1.machine_id

Let's use the decision tree on the **Activity (Factory Machine)** problem exactly as you would in an interview.

---

# Step 1: How many tables?

Given:

```text
Activity
```

Only **one table**.

✅ Decision:

```text
One table
```

↓

Move to the next question.

---

# Step 2: Do I need another row?

Look at one row.

| machine | process | type  | time  |
| ------- | ------- | ----- | ----- |
| 0       | 0       | start | 0.712 |

Question asks:

> Processing Time = End − Start

Can I calculate it from this row?

❌ No.

I need

| machine | process | type | time  |
| ------- | ------- | ---- | ----- |
| 0       | 0       | end  | 1.520 |

This is another row.

Decision:

```text
Need another row?
        YES
```

↓

Since it's the same table,

```text
SELF JOIN
```

---

# Step 3: Which row do I need?

Ask yourself:

For this row,

```text
Machine 0
Process 0
Start
```

Which row do I want?

Answer

```text
Machine 0
Process 0
End
```

Notice

Machine is same

Process is same

Only

```text
activity_type
```

changes.

---

# Step 4: How do I match them?

Question:

What uniquely identifies a process?

Suppose we only match on machine.

Machine 0 has

Process 0

Process 1

Wrong.

Need both

```text
machine_id

AND

process_id
```

Then ensure

One row is

```text
start
```

The other is

```text
end
```

So mentally

```text
Machine = Machine

Process = Process

Start ↔ End
```

---

# Step 5: What calculation?

Now the joined row becomes

| Machine | Process | Start | End   |
| ------- | ------- | ----- | ----- |
| 0       | 0       | 0.712 | 1.520 |

Now ask

What does the question want?

```text
End − Start
```

Easy.

---

# Step 6: Need aggregation?

Question says

> Find the average time **each machine** takes.

Words that should trigger you

```text
each machine
```

Immediately think

```text
GROUP BY machine_id
```

Average means

```text
AVG()
```

---

# Step 7: Rounding?

Question says

Round to

```text
3 decimal places
```

Think

```text
ROUND()
```

---

# Now write SQL

Now every clause comes naturally.

### Step A

Need two copies

```sql
FROM Activity a
JOIN Activity b
```

---

### Step B

Match same process

```sql
ON a.machine_id = b.machine_id
AND a.process_id = b.process_id
```

---

### Step C

Ensure

```text
a = start

b = end
```

So add

```sql
AND a.activity_type = 'start'
AND b.activity_type = 'end'
```

---

### Step D

Calculate processing time

```sql
b.timestamp - a.timestamp
```

---

### Step E

Average

```sql
AVG(...)
```

---

### Step F

Group

```sql
GROUP BY a.machine_id
```

---

### Step G

Round

```sql
ROUND(...,3)
```

---

# Final query

```sql
SELECT
    a.machine_id,
    ROUND(AVG(b.timestamp - a.timestamp), 3) AS processing_time
FROM Activity a
JOIN Activity b
ON a.machine_id = b.machine_id
AND a.process_id = b.process_id
AND a.activity_type = 'start'
AND b.activity_type = 'end'
GROUP BY a.machine_id;
```

---

## Notice something important

We didn't memorize anything.

We simply answered these questions one by one:

```
One table?
        ✔
Need another row?
        ✔
Same table?
        ✔
SELF JOIN
        ✔
How do I match?
machine + process
        ✔
What calculation?
end - start
        ✔
Need average?
AVG()
        ✔
Need per machine?
GROUP BY
        ✔
Need rounding?
ROUND()
```

This is exactly how experienced SQL developers think. Once you practice this process on 15–20 problems, you'll start recognizing the solution pattern within a minute instead of trying to recall syntax.


In [ ]:
-- LC 577
select e.name,b.bonus
from Employee e
left join Bonus b
on e.empId = b.empId
where bonus < 1000 or bonus is null

In [ ]:
Let's solve this **using the decision tree**, exactly like you should in an interview.

---

# Step 1: How many tables?

We have

```text
Employee

Bonus
```

✅ Two tables.

Move to the two-table decision tree.

---

# Step 2: Find the relationship

Look for the common column.

Employee

| empId | name |
| ----- | ---- |

Bonus

| empId | bonus |
| ----- | ----- |

The relationship is

```text
empId
```

So we know the join condition will eventually be

```sql
ON Employee.empId = Bonus.empId
```

---

# Step 3: Which table's rows should always appear?

This is the most important question.

Read the question again.

> Report the name and bonus amount of **each employee** who...

Notice it starts with

```text
each employee
```

Ask yourself:

If an employee **doesn't have a row** in the Bonus table, should we still consider them?

YES!

Because the question even says

> **did not get any bonus**

Those employees **won't even exist** in the Bonus table.

So if we use

```text
INNER JOIN
```

Brad and John disappear immediately.

That's wrong.

Decision:

```text
Need every employee

↓

LEFT JOIN
```

---

# Step 4: Visualize the LEFT JOIN

Employee

| empId | name   |
| ----- | ------ |
| 3     | Brad   |
| 1     | John   |
| 2     | Dan    |
| 4     | Thomas |

Bonus

| empId | bonus |
| ----- | ----- |
| 2     | 500   |
| 4     | 2000  |

After LEFT JOIN

| name   | bonus |
| ------ | ----- |
| Brad   | NULL  |
| John   | NULL  |
| Dan    | 500   |
| Thomas | 2000  |

Notice what happened.

Employees without bonuses still appear.

That's exactly what we wanted.

---

# Step 5: Read the conditions

Question says

Keep employees satisfying **either**

### Condition 1

```text
bonus < 1000
```

Who satisfies?

| name | bonus |
| ---- | ----- |
| Dan  | 500   |

---

### Condition 2

```text
No bonus
```

After LEFT JOIN

No bonus becomes

```text
NULL
```

Who satisfies?

| name | bonus |
| ---- | ----- |
| Brad | NULL  |
| John | NULL  |

---

# Step 6: Combine the conditions

The question says

> Either

That means

```text
OR
```

So mentally

```text
bonus < 1000

OR

bonus IS NULL
```

---

# Step 7: Return

Question asks for

```text
name

bonus
```

Nothing else.

---

# The Decision Tree Applied

```text
Two Tables
        │
        ▼
Relationship?

empId
        │
        ▼
Need all employees?

YES
        │
        ▼
LEFT JOIN
        │
        ▼
Need employees with

bonus <1000

OR

no bonus

        │
        ▼
WHERE

bonus <1000

OR

bonus IS NULL
        │
        ▼
Return

name

bonus
```

---

# Building the query step by step

### Step 1

Start from Employee because we need **every employee**.

```sql
FROM Employee e
```

---

### Step 2

Bring bonus information.

```sql
LEFT JOIN Bonus b
```

---

### Step 3

Match rows.

```sql
ON e.empId = b.empId
```

Now the table becomes

| name   | bonus |
| ------ | ----- |
| Brad   | NULL  |
| John   | NULL  |
| Dan    | 500   |
| Thomas | 2000  |

---

### Step 4

Apply the condition.

Need

```text
bonus <1000

OR

bonus IS NULL
```

---

### Step 5

Return

```text
name

bonus
```

---

# Final Query

```sql
SELECT e.name, b.bonus
FROM Employee e
LEFT JOIN Bonus b
ON e.empId = b.empId
WHERE b.bonus < 1000
   OR b.bonus IS NULL;
```

---

## A shortcut to recognize the correct join

When reading any SQL question, ask this one question:

> **"If there is no matching row in the second table, should this person still appear in the answer?"**

For this problem:

* Brad has **no bonus record**.
* John has **no bonus record**.

The question **wants both of them**.

So your answer should immediately be:

> "I need all employees, even if there's no matching bonus row."

That is the textbook situation for a **`LEFT JOIN`**. This single question will help you choose the correct join type in many interview problems.
